**Bài làm:** Nguyễn Bá Chính — 2A202602654 · K4-Track02-Day18

Đường lightweight; code gốc và assertion được giữ. Phân tích cuối notebook có hỗ trợ AI; số liệu lấy từ kernel thực thi. Xem `../AI_USAGE.md`.

In [1]:
import sys
from pathlib import Path
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "scripts" / "lakehouse.py").exists())
sys.path.insert(0, str(repo_root / "notebooks"))
sys.path.insert(0, str(repo_root / "scripts"))


# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [2]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [3]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 641389bc-00ce-4016-b447-7917a4… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ 9d12b2f7-5aec-4895-8462-ccf3c4… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [4]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
# The generator spans seven UTC days. Casting TIMESTAMPTZ to DATE otherwise
# uses the machine timezone and can split that interval into eight local dates.
con.execute("SET TimeZone='UTC'")
print("Aggregation timezone:", con.sql("SELECT current_setting('TimeZone')").fetchone()[0])
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Aggregation timezone: UTC


Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [5]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 7,
 'numFilesRemoved': 7,
 'filesAdded': '{"avg":2769.0,"max":2770,"min":2766,"totalFiles":7,"totalSize":19383}',
 'filesRemoved': '{"avg":2688.1428571428573,"max":2689,"min":2687,"totalFiles":7,"totalSize":18817}',
 'partitionsOptimized': 7,
 'numBatches': 7,
 'totalConsideredFiles': 7,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [6]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)
print("Gold full result (date/model/latency/cost/error):")
print(gold_df.sort(["date", "model"]).select(
    "date", "model", "p50_latency_ms", "p95_latency_ms", "cost_usd", "error_rate"
).write_csv())

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)
assert n_models == 3, "Gold must cover exactly the three input models"
assert gold_df.height == n_dates * n_models, "Gold is missing date/model groups"
assert gold_df.select("date", "model").n_unique() == gold_df.height
assert gold_df.filter(
    pl.any_horizontal(pl.col(c).is_null() for c in (
        "p50_latency_ms", "p95_latency_ms", "cost_usd", "error_rate"
    ))
).height == 0, "Gold metrics contain NULLs"
assert gold_df.filter(pl.col("p50_latency_ms") > pl.col("p95_latency_ms")).height == 0
assert gold_df.filter(pl.col("cost_usd") <= 0).height == 0
assert gold_df.filter(~pl.col("error_rate").is_between(0, 1)).height == 0
assert DeltaTable(SILVER).count() == con.sql("SELECT count(DISTINCT request_id) FROM silver").fetchone()[0]
assert all(Path(p).joinpath("_delta_log").exists() for p in (BRONZE, SILVER, GOLD))
print("[PASS] Gold complete date/model grid; non-null p50 <= p95; positive cost; error_rate in [0,1]")
print("[PASS] Silver request IDs unique; Bronze/Silver/Gold Delta logs present")

shape: (21, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-02 ┆ claude-hai ┆ 578.0      ┆ 1142.05   ┆ 16464541  ┆ 8270686   ┆ 0.049259  ┆ 46.254377 │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

## Số liệu kiểm chứng của bài nộp

In [7]:
import json as submission_json
submission_metrics = dict(bronze_rows=bronze_n, silver_rows=silver_n, duplicates_removed=bronze_n-silver_n,
             dates=n_dates, models=n_models, gold_rows=gold_df.height,
             gold=gold_df.sort(['date', 'model']).to_dicts())
print('SUBMISSION_METRICS_JSON=' + submission_json.dumps(submission_metrics, ensure_ascii=False, default=str))

SUBMISSION_METRICS_JSON={"bronze_rows": 200000, "silver_rows": 190052, "duplicates_removed": 9948, "dates": 7, "models": 3, "gold_rows": 21, "gold": [{"date": "2026-04-01", "model": "claude-haiku-4-5", "p50_latency_ms": 563.0, "p95_latency_ms": 1131.0, "total_prompt_tokens": "16722403", "total_completion_tokens": "8258111", "error_rate": 0.04882503348350176, "cost_usd": 46.4103664}, {"date": "2026-04-01", "model": "claude-opus-4-7", "p50_latency_ms": 3008.0, "p95_latency_ms": 6039.400000000001, "total_prompt_tokens": "5436642", "total_completion_tokens": "2709702", "error_rate": 0.05012996658002228, "cost_usd": 284.77728}, {"date": "2026-04-01", "model": "claude-sonnet-4-6", "p50_latency_ms": 1381.0, "p95_latency_ms": 2751.6499999999996, "total_prompt_tokens": "32948206", "total_completion_tokens": "16442926", "error_rate": 0.05046660117878193, "cost_usd": 345.488508}, {"date": "2026-04-02", "model": "claude-haiku-4-5", "p50_latency_ms": 578.0, "p95_latency_ms": 1142.0499999999993, "to

## Giải thích kết quả

Bronze 200,000 → Silver 190,052, loại 9,948 retry trùng request_id bằng ROW_NUMBER giữ timestamp sớm nhất. Gold có 21 nhóm = 7 ngày × 3 model. p50/p95 dùng QUANTILE_CONT trên latency; cost bằng tổng input/output token nhân đơn giá minh họa chia 1 triệu; error_rate là tỷ lệ status khác ok. Các assertion bổ sung kiểm tra toàn bộ grid, không NULL, p50≤p95, cost>0, error_rate∈[0,1], request_id duy nhất và ba Delta log trên đĩa. DuckDB đặt TimeZone=UTC trước CAST(ts AS DATE), khớp 7 ngày UTC của generator; timezone Asia/Bangkok mặc định chia khoảng này thành 8 ngày địa phương. Bảng giá/model là fixture của đề, không phải báo giá hay benchmark dịch vụ thật. Dữ liệu sinh trong lab là JSON hợp lệ; chưa chứng minh xử lý JSON sai cú pháp.